In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.dont_write_bytecode = True

# from src.main.python.engine import decode, prefill
from src.main.python.scheduler import scheduler
from src.main.python.config import config
from src.main.python.schema import model
import importlib

import torch
from transformers import GemmaTokenizerFast, BitsAndBytesConfig, Gemma3ForCausalLM, DynamicCache
PATH = "C://Users//user//LLM//smallgemma3"
# PATH = "D://LLM//small_gemma//gemma3_270M"

quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

llm_model = Gemma3ForCausalLM.from_pretrained(
    PATH,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
    )
llm_model = llm_model.eval()
tokenizer = GemmaTokenizerFast.from_pretrained(PATH)

# dev

In [ ]:
import gc
import torch
from typing import List, Dict, Any
from collections import OrderedDict
from src.main.python.schema import model 
from src.main.python.config import config
from src.main.python.engine import cache_manager
from src.main.python.engine import decode, prefill
from transformers import DynamicCache, Gemma3ForConditionalGeneration

class RequestManager:
    def __init__(self):
        self.PrefillList: Dict[model.Request.request_id, model.Request] = OrderedDict()
        self.DecodeList: Dict[model.Request.request_id, model.Request] = OrderedDict()

    def add_request(self, request: model.Request):
        if request.status is model.RequestStatus.PREFILLING:
            self.PrefillList[request.request_id] = request
        elif request.status is model.RequestStatus.DECODING:
            self.DecodeList[request.request_id] = request

    def step(self):
        self.idx = list() # 之後拿來更新cache
        d_input_ids = list()
        d_caches = list()
        p_input_ids = list()
        p_caches = list()
        # ----- 先取input_ids(同時更新狀態)，再放回List中 ----- #

        # -Decoding- #
        decoding_requests: List[model.Request] = list()
        for _ in range(min([config.BATCH_SIZE, len(self.DecodeList)])):
            key, value = self.DecodeList.popitem(last=False)
            decoding_requests += [value]
            d_input_ids += [value.get_ids()]
            d_caches += [value.kv_cache]
        
        # -Prefilling- #
        for _ in range(min([config.BATCH_SIZE - len(d_input_ids), len(self.PrefillList)])):
            if len(self.PrefillList) == 0:
                break
            key, value = self.PrefillList.popitem(last=False)
            ids = value.get_ids()
            if len(ids) < config.PREFILL_TOKEN_SIZE:
                zeros = torch.zeros((1, config.PREFILL_TOKEN_SIZE - ids.shape[1]), dtype=torch.long)
                ids = torch.cat([ids, zeros], dim=1)
            p_input_ids += [ids]
            p_caches += [value.kv_cache]
            self.idx += [(value.status, key)]
            self.PrefillList[key] = value
        return d_input_ids, d_caches, p_input_ids, p_caches, decoding_requests # 多輸出 decoding_requests，讓外面可以接續生成
    
    def update(self, caches: List[DynamicCache]):
        # ----- 更新cache ----- #
        i = 0
        for status, _id in self.idx:
            if status is model.RequestStatus.PREFILLING:
                self.PrefillList[_id].kv_cache = caches[i]

            # ----- 從update手段去啟動Decoding的轉移函數 ----- #
            elif status is model.RequestStatus.DECODING: # δ(PREFILLING, Prefilling_Complete) = Decoding
                decoding = self.PrefillList.pop(_id)
                self.DecodeList[_id] = decoding
                self.DecodeList[_id].kv_cache = caches[i]
            i += 1

In [4]:
import gc
import torch
from typing import List, Dict, Any
from src.main.python.schema import model 
from src.main.python.config import config
from src.main.python.engine import cache_manager
from transformers import DynamicCache, Gemma3ForConditionalGeneration

def prefill_infer(model: Gemma3ForConditionalGeneration,  
          input_ids: List[torch.Tensor], 
          kv_caches: List[DynamicCache]):
    if len(input_ids) == 0:
        return None, []
    try:
        cache = None

        # ----- 整合所有input ----- #
        _input_ids = torch.cat(input_ids, dim=0)
        cache = cache_manager.KVCache_merge(kv_caches)

        # ----- Prefilling過程 ----- #
        with torch.no_grad():
            model(
                input_ids=torch.LongTensor(_input_ids).to(model.device),
                use_cache=True,
                past_key_values=cache,
                )
        print("Prefill Cache Shape:", cache.key_cache[0].shape)
        # ----- 拆解cache ----- #
        eds = []
        for ids in input_ids:
            _list = torch.where(ids==0)[1].tolist()
            if _list:
                eds += [_list[0] - ids.shape[1]]
            else:
                eds += [None]
        caches = cache_manager.KVCache_split(cache,eds)
    finally:
        for item in ("input_ids", ):
            exec(f"del {item}")
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return None, caches


In [5]:
import gc
import torch
from typing import List, Dict, Any
from src.main.python.schema import model 
from src.main.python.config import config
from src.main.python.engine import cache_manager
from transformers import DynamicCache, Gemma3ForConditionalGeneration
MAX_NEW_TOKENS_SIZE = 16

def decode_infer(model: Gemma3ForConditionalGeneration,       
                 input_ids: List[torch.Tensor], 
                 kv_caches: List[DynamicCache],
                 uids: List[str],
                 ):
    if len(input_ids) == 0:
        return None, []
    try:
        # ----- 宣告物件 ----- #
        device = model.device
        n_batch = len(kv_caches)
        eos_token_ids = [1, 106] # processor.tokenizer.eos_token_id == 1
        unfinished_sequences = torch.ones(n_batch, dtype=torch.long, device=device)
        generated_ids = {uid: list() for uid in uids}
        merged_cache = cache_manager.KVCache_merge(kv_caches)
        print("Decode Cache Shape:", merged_cache.key_cache[0].shape)

        # ----- 將input做合併 ----- #
        input_ids_tensor = torch.cat(input_ids, dim=0).to(device)

        # ----- Decoding Loop ----- #
        for step in range(MAX_NEW_TOKENS_SIZE):

            # ----- 全部都做完了 ----- #
            if unfinished_sequences.max() == 0:
                break # Stop Decoding
            
            # ----- 計算position_ids ----- #
            cache_len = merged_cache.get_seq_length(layer_idx=0)
            position_ids = torch.tensor([[cache_len-1]], device=device).expand(n_batch, -1)

            # ----- 生成tokens ----- #
            with torch.no_grad():
                outputs = model(
                    input_ids=input_ids_tensor,
                    past_key_values=merged_cache,
                    position_ids=position_ids,
                    use_cache=True)
            logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(logits, dim=-1)
            for i in range(n_batch):
                if unfinished_sequences[i]:
                    generated_ids[uids[i]].append(next_token[i].item())
            input_ids_tensor = next_token.unsqueeze(1)
            is_eos = torch.isin(next_token, torch.tensor(eos_token_ids, device=device))
            unfinished_sequences.mul_(~is_eos) # in-place更新

        # ----- 找EOS位置 ----- #
        eds = list()
        for i in range(n_batch):
            generated_length = len(generated_ids[uids[i]])
            _idx = generated_length - MAX_NEW_TOKENS_SIZE
            eds += [_idx if _idx < 0 else None]

        # ----- Cache更新 ----- #
        new_caches_list = cache_manager.KVCache_split(merged_cache, eds)
 
    finally:
        for item in ("input_ids", "outputs", "logits", "next_token", "token_id"):
            try:
                exec(f"del {item}")
            except:
                pass
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    return generated_ids, new_caches_list

# 測試

In [3]:
import uuid
import torch
import importlib
from src.main.python.engine import cache_manager, decode, prefill

In [4]:
# importlib.reload(scheduler)
MSG = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""
TEXT = dict()
_scheduler = scheduler.RequestManager()
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    ids = tokenizer.encode(MSG.format(prompt=sentences))
    request = model.Request(input_ids = torch.tensor(ids).unsqueeze(0),
                            status = model.RequestStatus.PREFILLING,
                            request_id = str(uuid.uuid4()),
                            kv_cache = DynamicCache()
            )
    # print("token長度:", len(ids))
    _scheduler.add_request(request)

    for _ in range(8):
        d_input_ids, d_caches, p_input_ids, p_caches, decode_requests = _scheduler.step()
        text, DCACHES = decode.infer(llm_model, d_input_ids, d_caches, [r.request_id for r in decode_requests])
        _, PCACHES = prefill.infer(llm_model, p_input_ids, p_caches)
        _scheduler.update(PCACHES)
        if text is not None:
            for i, r in enumerate(decode_requests):
                _id = r.request_id
                r.input_ids = torch.tensor(text[_id][-1:]).unsqueeze(0)
                print(_id, tokenizer.decode(text[_id][:]))
                r.kv_cache = DCACHES[i]
                _scheduler.add_request(r)
                TEXT[_id] = TEXT.get(_id, "") + tokenizer.decode(text[_id], skip_special_tokens=True)
    print("----- 最後輸出 -----\n", TEXT, "\n", "-"*50)

`cache.key_cache[idx]` is deprecated and will be removed in v4.56.0. Use `cache.layers[idx].keys` instead.
`cache.value_cache[idx]` is deprecated and will be removed in v4.56.0. Use `cache.layers[idx].values` instead.


bbdb4eb6-5cc1-4f53-90e3-7773b73232ac 半導體廠務通常在做以下幾件事：

1.  
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac **安全確保：**
   -   安全是首要的。安全
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac 措施包括：
       -   嚴格的防護措施，包括：
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac 
           -   使用安全警示燈
           -   使用安全警
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac 示警告燈
           -   使用安全警示警告燈
   -
----- 最後輸出 -----
 {'bbdb4eb6-5cc1-4f53-90e3-7773b73232ac': '半導體廠務通常在做以下幾件事：\n\n1.  **安全確保：**\n   -   安全是首要的。安全措施包括：\n       -   嚴格的防護措施，包括：\n           -   使用安全警示燈\n           -   使用安全警示警告燈\n           -   使用安全警示警告燈\n   -'} 
 --------------------------------------------------
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac    安全措施包括：
       -   使用安全警示燈
       
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac -   使用安全警示警告燈
       -   使用安全警示
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac 警告燈
       -   使用安全警示警告燈
       -   
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac 使用安全警告燈
       -   使用安全警告燈
       -   
bbdb4eb6-5cc1-4f53-90e3-7773b73232ac 使用安全警告燈
       -   使用安全警告燈
       -   
b021142d-c

KeyboardInterrupt: 

In [13]:
tokenizer.decode([106])

'<end_of_turn>'

In [7]:
torch.tensor(ids).unsqueeze(0).shape

torch.Size([1, 25])

In [7]:
for t in TEXT:
    print(t, "\n", TEXT[t], "\n","-"*50,"\n")

b8b1206f-b3b6-4273-8466-0fe028248ba0 
 半導體廠務主要做的事情非常複雜，可以概括為以下幾個關鍵步驟，大致可以分為以下幾個階段：

**1. 設計與製程規劃 (Design & Process Planning):**

* **晶片設計 (Chip Design):**  這是最核心的部分，由設計工程師使用專業軟體（例如Cadence、Synopsys）設計晶片內部電路的結構，包括邏輯電路、記憶體、處理器等等。
* **製程規劃 (Process Planning):**  根據設計，工程師會制定詳細的製造流程，包括使用的材料、設備、參數設定等，確保晶片能夠按照設計正確地製造出來。
* **模擬與驗證 (Simulation & Verification):**  在實際製造之前，會使用模擬軟體驗證設計的正確性，並預測晶片在實際製造中的表現。


**2. 製造 (Fabrication - 晶圓製造):**

這是半導體廠務最複雜、最昂貴的部分，主要包含以下幾個階段：

* **晶圓切割 (Wafer Fabrication):**  使用高純度的矽晶圓作為基底，進行切割。
* **薄膜堆疊 (Thin Film Deposition):**  利用各種技術（例如化學氣相沉積、物理氣相沉積）在晶圓上 depositing 不同的薄膜材料，形成電路元件。
* **光刻 (Photolithography):**  使用光刻機將設計圖案轉印到晶圓上，這是製造複雜電路的關鍵步驟。
* **蝕刻 (Etching):**  利用化學或物理方法去除晶圓上不需要的部分，形成電路圖案。
* **金屬化 (Metallization):**  在晶圓上 depositing 金屬層，連接不同的電路元件。
* **測試 (Testing):**  在晶圓上進行電氣測試，驗證每個電路元件的功能是否正常。


**3. 測試與封裝 (Testing & Packaging):**

* **晶圓測試 (Wafer Probe):**  使用自動測試設備（ATE）對晶圓上的每個電路進行測試，找出缺陷 
 -------------------------------------------------- 

aff802e1-1813-4e5c-95

# 服務開發

In [1]:
import os
import io
import sys
import uuid
import time
import json
import torch
import queue
import asyncio
import uvicorn
import importlib
import threading
import subprocess
import multiprocessing

In [2]:
from src.main.python.schema import model
from src.main.python.config import config
from src.main.python.scheduler import scheduler
from src.main.python.engine import decode, prefill

from transformers import GemmaTokenizerFast, BitsAndBytesConfig, Gemma3ForCausalLM, DynamicCache

c:\Users\user\anaconda3\envs\dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
TEXT = dict()
MSG = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""
Event = multiprocessing.Event()
manager = multiprocessing.Manager()
CacheDict = manager.dict()
TaskQueue = manager.Queue()
ResDict = manager.dict()

PATH = "D://LLM//gemma//gemma3_4b"
tokenizer = GemmaTokenizerFast.from_pretrained(PATH)

In [4]:
def Inference_Engine():
    _scheduler = scheduler.RequestManager()
    quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )

    llm_model = Gemma3ForCausalLM.from_pretrained(
        PATH,
        quantization_config=quantization_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    llm_model = llm_model.eval()

    for tiral in range(10):
        if not TaskQueue.empty():
            ids, request_id = TaskQueue.get_nowait()
            Cache = CacheDict.get(request_id, DynamicCache())
            request = model.Request(
                                input_ids = torch.tensor(ids).unsqueeze(0),
                                status = model.RequestStatus.PREFILLING,
                                request_id = request_id,
                                kv_cache = Cache
            )
            _scheduler.add_request(request)
        d_input_ids, d_caches, p_input_ids, p_caches, decode_requests = _scheduler.step()
        UUID = [r.request_id for r in decode_requests]
        TEXT, DCACHES = decode.infer(llm_model, d_input_ids, d_caches, UUID)
        _, PCACHES = prefill.infer(llm_model, p_input_ids, p_caches)
        _scheduler.update(PCACHES)

        if TEXT is not None:
            for i, uid in enumerate(UUID):
                ResDict[uid] = TEXT[uid]
                CacheDict[uid] = DCACHES[i]

In [5]:
def RequestsHandler(uid):
    text = ""
    while "<end_of_turn>" not in text:
        ids = ResDict[uid]
        input_ids = torch.tensor(ids[-1:]).unsqueeze(0)
        TaskQueue.put((input_ids, uid))
        text += tokenizer.decode(ids, skip_special_tokens=True)
        yield text

In [6]:
uid = str(uuid.uuid4())
prompt = "你好，你是誰 ?"
if uid not in CacheDict:
    uid = str(uuid.uuid4())
Cache = CacheDict.get(uid, DynamicCache())
ids = tokenizer.encode(MSG.format(prompt=prompt))
TaskQueue.put((ids, uid))

In [7]:
Inference_Engine()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

`cache.key_cache[idx]` is deprecated and will be removed in v4.56.0. Use `cache.layers[idx].keys` instead.
`cache.value_cache[idx]` is deprecated and will be removed in v4.56.0. Use `cache.layers[idx].values` instead.


In [9]:
tokenizer.decode(ResDict[uid])

'您好！我是由 Google 訓練的大型語言模型。您可以把我當'

In [8]:
TaskQueue.get_nowait()

Empty: 

# 單一函數操作

In [35]:
msg = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""

def gemma3_resp(prompt):
    max_seq_len = 64

    # ----- 結果儲存 ----- #
    res = list()

    # ----- Prompt token產生 ----- #
    MSG = msg.format(prompt=prompt)
    input_ids = torch.tensor(tokenizer.encode(MSG)).to(llm_model.device)
    input_ids = input_ids.unsqueeze(0)
    eos_token_ids = [tokenizer.eos_token_id, 106]

    # ----- Cache宣告 ----- #
    past_key_values = DynamicCache()
    
    # ----- Prefill ----- #
    chunks = torch.split(input_ids[:, :-1], 32, dim=-1)
    st = 0
    ed = 0
    with torch.no_grad():
        for chunk in chunks:
            ed = st + chunk.shape[1]
            llm_model(input_ids=chunk, use_cache=True, past_key_values=past_key_values)
            st = ed
    
    # ----- Auto Regressive生成 ----- #
    input_ids = input_ids[:, -1:]
    try:
        for _ in range(max_seq_len):
            with torch.no_grad():
                # ----- Update position ----- #
                ed += 1

                # ----- Update model kwargs ----- #
                # cache_position = torch.arange(ed-1, ed, dtype=torch.long, device = llm_model.device)
                cache_position = torch.arange(past_key_values.get_seq_length(layer_idx=0)-1, 
                                              past_key_values.get_seq_length(layer_idx=0), 
                                              dtype=torch.long, 
                                              device = llm_model.device)
                # ----- 生成token ----- #
                outputs = llm_model(input_ids=input_ids, use_cache=True, past_key_values=past_key_values, cache_position=cache_position)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                token_id = next_token.item()
                input_ids = next_token

                # ----- 判斷是否終止 ----- #
                if token_id in eos_token_ids:
                    break

                # ----- 紀錄token ----- #
                res += [tokenizer.decode(token_id)]
                
                # ----- 輸出文字字串 ----- #
                # print(res[-1], end="", flush=True)
    except:
        for item in ("input_ids", "outputs", "ogits", "next_token", "token_id"):
            try:
                eval(f"del {item}")
            except:
                pass
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
    return "".join(res), past_key_values

In [36]:
for sentences in ("半導體廠務通常在做什麼", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理，它裡面有很多關於統計觀念，也請一一闡述", "什麼是巨單交易", "什麼是機器學習"):
    text1, cache1 = gemma3_resp(sentences)
    print(cache1.key_cache[0].shape)
    print(text1, "\n", "-"*50)

torch.Size([1, 1, 92, 256])
半導體廠務通常在做以下幾個核心任務：

1. **硬件製造：**
   *   製造各種半導體晶片 (如晶片、晶片組件、晶片組件、晶片組件、晶片組裝、晶片組裝、晶片 
 --------------------------------------------------
torch.Size([1, 1, 113, 256])
Black Scholes的核心精神是：**“黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑色的黑 
 --------------------------------------------------
torch.Size([1, 1, 88, 256])
巨單交易是指在一個交易中，**所有的交易者（包括交易员）都同意在交易中**，**在交易中，**所有的交易者（包括交易员）都同意**，**交易者（包括交易员）**，**在交易中**，**交易者（包括 
 --------------------------------------------------
torch.Size([1, 1, 87, 256])
機器學習 (Machine Learning) 是一種人工智能 (Artificial Intelligence) 的方法，它利用數據 (Data) 來學習和預測模式 (Patterns) 和決斷 (Predictions) 的能力。

**核心概念：**

*   **數據 (Data):**  機器學習需要收集大量的數據，包含 
 --------------------------------------------------
